In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from datasets import load_dataset
from copy import deepcopy

class MASTrainer(Trainer):
    def __init__(self, *args, old_params=None, importance=None, mas_lambda=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.old_params = old_params
        self.importance = importance
        self.mas_lambda = mas_lambda

    def compute_loss(self, model, inputs, return_outputs=False):
        # Standard loss
        outputs = model(**inputs)
        loss = outputs.loss

        # MAS regularization
        if self.old_params and self.importance:
            mas_reg = 0.0
            for name, param in model.named_parameters():
                if name in self.old_params:
                    mas_reg += (self.importance[name] * (param - self.old_params[name]).pow(2)).sum()
            loss += self.mas_lambda * mas_reg

        return (loss, outputs) if return_outputs else loss

def compute_mas_importance(model, dataloader, device):
    importance = {}
    model.eval()
    for name, param in model.named_parameters():
        importance[name] = torch.zeros_like(param)

    for batch in dataloader:
        model.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # Forward pass
        output = model.encoder(input_ids=input_ids, attention_mask=attention_mask)[0]
        loss = output.norm(2)
        loss.backward()

        # Accumulate importance
        for name, param in model.named_parameters():
            if param.grad is not None:
                importance[name] += param.grad.abs().detach()

    # Normalize
    for name in importance:
        importance[name] /= len(dataloader)

    return importance

def freeze_old_params(model):
    return {name: param.detach().clone() for name, param in model.named_parameters()}

# Example usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

def preprocess_function(examples):
    inputs = [ex["input"] for ex in examples]
    targets = [ex["target"] for ex in examples]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, return_tensors="pt", max_length=128)
    labels = tokenizer(targets, padding="max_length", truncation=True, return_tensors="pt", max_length=128).input_ids
    model_inputs["labels"] = labels
    return model_inputs

# Load a task dataset (assumed to be in text2text format)
task1 = load_dataset("your_task1_dataset")["train"].map(preprocess_function, batched=True)
task2 = load_dataset("your_task2_dataset")["train"].map(preprocess_function, batched=True)

dataloader = DataLoader(task1, batch_size=8)

# === Train on Task 1 ===
training_args = TrainingArguments(
    output_dir="./flan_task1",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_strategy="no",
    logging_steps=10,
)
trainer = Trainer(model=model, args=training_args, train_dataset=task1)
trainer.train()

# === Compute MAS importance ===
importance = compute_mas_importance(model, dataloader, device)

# === Save frozen params ===
old_params = freeze_old_params(model)

# === Train on Task 2 with MAS ===
trainer2 = MASTrainer(
    model=model,
    args=TrainingArguments(output_dir="./flan_task2", per_device_train_batch_size=8, num_train_epochs=3),
    train_dataset=task2,
    old_params=old_params,
    importance=importance,
    mas_lambda=1.0,
)
trainer2.train()
